In [4]:
import pandas as pd
import clickhouse_connect
from datetime import datetime





import os
from dotenv import load_dotenv

load_dotenv()


True

In [5]:

# создаём клиент
# client = clickhouse_connect.get_client(
#     host= os.getenv("CLICKHOUSE_HOST") ,   # замени на свой хост/адрес
#     port= os.getenv("CLICKHOUSE_PORT"),
#     username=  os.getenv("CLICKHOUSE_USER"),
#     password= os.getenv("CLICKHOUSE_PASSWORD")
# )
client = clickhouse_connect.get_client(
    host="84.201.160.255",   # замени на свой хост/адрес
    port=8123,
    username="peter",
    password="1234"
)

##### заносим в таблицу email_unique только уникальные, очищенные от повторов письма (посленее письмо в цепочке) чтобы уменьшить количество писем для обработки

In [6]:
import pandas as pd

CHUNK_SIZE = 20000


def dedup_thread(df_thread):

    rows = df_thread.sort_values(
        by="body_text",
        key=lambda x: x.str.len(),
        ascending=False
    )

    kept = []

    for _, row in rows.iterrows():

        body = row["body_text"]

        duplicate = False

        for k in kept:
            if body in k["body_text"]:
                duplicate = True
                break

        if not duplicate:
            kept.append(row)

    return pd.DataFrame(kept)


offset = 0

while True:

    query = f"""
    SELECT
        id,
        thread_key,
        message_id,
        subject,
        from_addr,
        to_addr,
        sent_at_utc,
        folder,
        body_text
    FROM mailkb.emails
    WHERE body_text IS NOT NULL
    ORDER BY thread_key, sent_at_utc
    LIMIT {CHUNK_SIZE}
    OFFSET {offset}
    """

    df = client.query_df(query)

    if df.empty:
        break

    print("Loaded rows:", len(df))

    result = []

    for thread_key, df_thread in df.groupby("thread_key"):

        deduped = dedup_thread(df_thread)

        result.append(deduped)

    df_result = pd.concat(result)

    rows = df_result.values.tolist()

    client.insert(
        "mailkb.emails_unique",
        rows,
        column_names=df_result.columns.tolist()
    )

    print("Inserted:", len(rows))

    offset += CHUNK_SIZE

Loaded rows: 20000
Inserted: 17013
Loaded rows: 6675
Inserted: 5681
